<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_3/bonus/pandas_data_analysis/note_bonus_pandas_foundation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ☁️ Colab / новий комп'ютер: завантажити папку уроку з даними й встановити пакети
import importlib.util, os, subprocess, sys

LESSON = "module_3/bonus/pandas_data_analysis"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"
if not os.path.isdir("data"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

packages = {"pandas": "pandas", "matplotlib": "matplotlib", "seaborn": "seaborn", "plotly": "plotly",
            "scipy": "scipy", "pyarrow": "pyarrow"}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("папка:", os.getcwd(), "| даних:", len(os.listdir("data")))

папка: C:\Users\victo\PycharmProjects\PY-Course-Victor-Nikoriak-22-09-2026\module_3\bonus\pandas_data_analysis | даних: 12


# 📊 Урок 52: Pandas — Фундамент аналізу даних

> **Мета уроку:** Навчитися «думати таблицями» — це фундаментальний крок для будь-якого аналітика або інженера даних. У світі `pandas` кожна проблема зводиться до трансформації колонок і рядків.

---

## 🗺️ Зміст уроку

1. Що таке DataFrame — модель реального світу
2. Архітектура Pandas під капотом
3. Управління пам'яттю та типи даних
4. Завантаження CSV-файлів
5. Огляд даних: head, info, describe
6. Фільтрація: loc та query
7. Групування та агрегації
8. Робота з датами (datetime)
9. Об'єднання датасетів — merge та concat

---

**Датасети, які використовуємо:**
- 🌾 `wfp_food_prices_ukr.csv` — ціни на продукти харчування в Україні (WFP)
- 💱 `exchange-rates_ukr.csv` — курс гривні до долара (FAO)
- 📉 `poverty_ukr.csv` — соціально-економічні індикатори
- 🌍 `global-market-monitor.csv` — глобальний моніторинг продовольчих ринків

---

## 1. Що таке DataFrame — модель реального світу 🌍 <a id='1-dataframe'></a>

### Концепція: таблиця як знімок реальності

У `pandas` **DataFrame** — це двовимірна структура даних, яка логічно є аналогом:
- електронної таблиці Excel
- реляційної таблиці SQL
- JSON-об'єктів у вигляді списку словників

Але думати про DataFrame треба глибше — це **модель реального світу**:

| Реальний світ | DataFrame |
|---|---|
| Одна подія, спостереження, об'єкт | **Рядок (row)** |
| Атрибут або характеристика об'єкта | **Стовпець (column)** |
| Унікальний ідентифікатор об'єкта | **Індекс (index)** |

### Архітектура: DataFrame = словник із Series

```
DataFrame
├── Index (індекс рядків) ← спільний для всіх стовпців
├── Column "date"    → Series (dtype: datetime64)
├── Column "market"  → Series (dtype: object / string)
├── Column "price"   → Series (dtype: float64)
└── Column "region"  → Series (dtype: category)
```

**Series** — базовий будівельний блок. Це одновимірний масив даних, **жорстко пов'язаний з об'єктом `Index`**. Концептуально `Series` нагадує словник Python: мітки індексу — це ключі, дані — це значення.

### Ключова різниця: Series vs DataFrame

| | Series | DataFrame |
|---|---|---|
| Вимірність | 1D (одна колонка) | 2D (таблиця) |
| Однорідність | Всі елементи одного типу | Кожна колонка може мати свій тип |
| `[]` оператор | Вибирає **рядки** | Вибирає **стовпці** |
| `.loc[]` | Один аргумент (рядок) | Два аргументи `[рядки, стовпці]` |
| Data alignment | По мітках рядків | По рядках AND по стовпцях одночасно |

> 💡 **Увага:** `df['a']` поверне стовпець `'a'`. Якщо хочете рядок — використовуйте `df.loc['a']`!

### Автоматичне вирівнювання даних (Data Alignment)

Це фундаментальна особливість Pandas: при операціях над кількома об'єктами дані зіставляються **за мітками індексу**, а не за позицією в масиві. Якщо мітка є в одному об'єкті, але відсутня в іншому — система автоматично підставляє `NaN` (аналог SQL `FULL OUTER JOIN`).

In [1]:
import pandas as pd
import numpy as np

# --- Демонстрація: Series як словник ---
print("=== Series: одновимірний масив з індексом ===")
prices_series = pd.Series(
    data=[45.5, 38.2, 22.1, 67.8],
    index=['Київ', 'Львів', 'Харків', 'Одеса'],
    name='Ціна хліба (UAH)'
)
print(prices_series)
print(f"\nТип даних: {prices_series.dtype}")
print(f"Ціна у Львові: {prices_series['Львів']} UAH")

=== Series: одновимірний масив з індексом ===
Київ      45.5
Львів     38.2
Харків    22.1
Одеса     67.8
Name: Ціна хліба (UAH), dtype: float64

Тип даних: float64
Ціна у Львові: 38.2 UAH


In [6]:
# --- Демонстрація: DataFrame = колекція Series ---
print("=== DataFrame: двовимірна таблиця ===")
data = {
    'місто': ['Київ', 'Львів', 'Харків', 'Одеса'],
    'ціна_хліб': [45.5, 38.2, 22.1, 67.8],
    'ціна_молоко': [34.0, 32.5, 29.0, 38.5],
    'регіон': ['Центр', 'Захід', 'Схід', 'Південь']
}
df_demo = pd.DataFrame(data)
print(df_demo)
print(f"\nТипи: {df_demo.dtypes.to_dict()}")
print(f"\ndf_demo['ціна_хліб'] — це Series:")
print(type(df_demo['ціна_хліб']))

=== DataFrame: двовимірна таблиця ===
    місто  ціна_хліб  ціна_молоко   регіон
0    Київ       45.5         34.0    Центр
1   Львів       38.2         32.5    Захід
2  Харків       22.1         29.0     Схід
3   Одеса       67.8         38.5  Південь

Типи: {'місто': <StringDtype(na_value=nan)>, 'ціна_хліб': dtype('float64'), 'ціна_молоко': dtype('float64'), 'регіон': <StringDtype(na_value=nan)>}

df_demo['ціна_хліб'] — це Series:
<class 'pandas.Series'>


In [7]:
# --- Демонстрація: автоматичне вирівнювання даних ---
print("=== Автоматичне вирівнювання (Data Alignment) ===")
s1 = pd.Series([10, 20, 30], index=['a', 'b', 'c'])
s2 = pd.Series([100, 200, 300], index=['b', 'c', 'd'])  # 'd' є тільки в s2, 'a' — тільки в s1

print("s1:", s1.to_dict())
print("s2:", s2.to_dict())
print("\ns1 + s2 (автоматичне вирівнювання, аналог FULL OUTER JOIN):")
print(s1 + s2)
print("\n💡 'a' і 'd' → NaN, бо вони є тільки в одному з об'єктів")

=== Автоматичне вирівнювання (Data Alignment) ===
s1: {'a': 10, 'b': 20, 'c': 30}
s2: {'b': 100, 'c': 200, 'd': 300}

s1 + s2 (автоматичне вирівнювання, аналог FULL OUTER JOIN):
a      NaN
b    120.0
c    230.0
d      NaN
dtype: float64

💡 'a' і 'd' → NaN, бо вони є тільки в одному з об'єктів


---

## 2. Архітектура Pandas під капотом 🏗️ <a id='2-architecture'></a>

Уяви Pandas як **висококорівневу обгортку над низькорівневими масивами**, створену спеціально для аналізу гетерогенних (різнотипних) даних.

```
┌─────────────────────────────────────────┐
│         Рівень I/O                      │  read_csv, read_sql, to_excel...
├─────────────────────────────────────────┤
│    Рівень структур даних (Ядро)         │  Series, DataFrame, Index
├─────────────────────────────────────────┤
│      Рівень типів даних (dtypes)        │  int64, float32, category, datetime64
├─────────────────────────────────────────┤
│  Фундаментальний шар (Backend)          │  NumPy  /  Apache Arrow (PyArrow)
└─────────────────────────────────────────┘
```

### Модель виконання: Eager Execution

На відміну від SQL або Dask, Pandas виконує команди **негайно** (eager execution). Це дає розробнику повний контроль, але вимагає самостійної оптимізації коду.

### Обмеження: In-Memory

Дані повністю завантажуються в оперативну пам'ять. Автор бібліотеки рекомендує мати RAM у **5–10 разів більше**, ніж розмір датасету на диску. При перевищенні ліміту переходять на:
- **Chunking** — обробка по частинах
- **Dask** — паралельні обчислення
- **PySpark** — розподілені кластери

In [2]:
# --- Перевірка версії та бекенду ---
print(f"Pandas версія: {pd.__version__}")
print(f"NumPy версія: {np.__version__}")

# Перевіримо, чи доступний PyArrow
try:
    import pyarrow as pa
    print(f"PyArrow версія: {pa.__version__} ✅")
except ImportError:
    print("PyArrow не встановлено (встановіть: pip install pyarrow)")

Pandas версія: 2.3.3
NumPy версія: 2.2.6
PyArrow версія: 23.0.1 ✅


---

## 3. Управління пам'яттю та типи даних 💾 <a id='3-memory'></a>

Оскільки Pandas — це in-memory система, правильний вибір типів даних критично важливий.

### NumPy vs PyArrow — два бекенди

| | NumPy (класичний) | PyArrow (сучасний, Pandas ≥ 2.0) |
|---|---|---|
| Рядки | `object` (неефективно!) | `string[pyarrow]` (компактно) |
| Цілі числа з NaN | конвертує в `float64` 😱 | `int64[pyarrow]` (NaN-safe) |
| Пропущені значення | `np.nan` | `pd.NA` |
| Продуктивність | baseline | значно швидше на рядках |

### Практичні оптимізації

1. **Downcasting** — замість `int64` (8 байт) → `int8` (1 байт), якщо значення в діапазоні -128..127
2. **Category** — для текстових стовпців з низькою кардинальністю (регіони, типи товарів). Економія до 90% пам'яті!
3. **datetime64** замість рядків для дат

In [4]:
# --- Демонстрація економії пам'яті ---

# Симулюємо невеликий датасет з повторюваними категоріями
n = 10_000
df_mem = pd.DataFrame({
    'регіон': np.random.choice(['Київська', 'Львівська', 'Харківська', 'Одеська', 'Дніпропетровська'], n),
    'товар': np.random.choice(['Хліб', 'Молоко', 'Картопля', 'Цибуля'], n),
    'ціна': np.random.uniform(5, 200, n).astype('float64'),
    'кількість': np.random.randint(1, 100, n).astype('int64')
})


In [5]:

print("=== ДО оптимізації ===")
print(df_mem.dtypes)
mem_before = df_mem.memory_usage(deep=True).sum()
print(f"\nПам'ять: {mem_before:,} байт ({mem_before / 1024:.1f} KB)")

=== ДО оптимізації ===
регіон        object
товар         object
ціна         float64
кількість      int64
dtype: object

Пам'ять: 2,120,982 байт (2071.3 KB)


In [7]:
# --- Оптимізуємо типи ---
df_opt = df_mem.copy()

# Текстові колонки з повторами → category
df_opt['регіон'] = df_opt['регіон'].astype('category')
df_opt['товар'] = df_opt['товар'].astype('category')

In [8]:
# Ціна: float32 замість float64 (вдвічі менше)
df_opt['ціна'] = df_opt['ціна'].astype('float32')

# Кількість: int8 (0-100 поміщається в int8, але int16 безпечніше)
df_opt['кількість'] = df_opt['кількість'].astype('int16')

In [9]:
print("=== ПІСЛЯ оптимізації ===")
print(df_opt.dtypes)
mem_after = df_opt.memory_usage(deep=True).sum()
print(f"\nПам'ять: {mem_after:,} байт ({mem_after / 1024:.1f} KB)")
print(f"\n💾 Економія: {(1 - mem_after/mem_before)*100:.1f}%")

=== ПІСЛЯ оптимізації ===
регіон       category
товар        category
ціна          float32
кількість       int16
dtype: object

Пам'ять: 81,515 байт (79.6 KB)

💾 Економія: 96.2%


---

## 4. Завантаження CSV-файлів 📂 <a id='4-loading'></a>

### `pd.read_csv()` — I/O Readers

Pandas абстрагує від складнощів парсингу. Але **думати таблицями треба вже на етапі завантаження** — правильні параметри зекономлять пам'ять та час.

| Параметр | Призначення |
|---|---|
| `sep` | Розділювач (`,`, `;`, `\t`) |
| `usecols` | Завантажити лише потрібні стовпці |
| `parse_dates` | Автоматично розпізнати дати |
| `dtype` | Вказати типи заздалегідь (уникнути зайвого перетворення) |
| `nrows` | Завантажити лише N рядків (для попереднього огляду) |

> 🎯 **Правило:** Завантажуй лише ті колонки, що потрібні (`usecols`). Для датасетів >1 млн рядків це критично.

In [14]:
import os

# Шлях до директорії з даними (відносно ноутбука)
DATA_DIR = 'data'

# --- 1. Завантажуємо ціни на продукти харчування ---
# Спочатку подивимося, скільки рядків у файлі
df_food_preview = pd.read_csv(os.path.join(DATA_DIR, 'wfp_food_prices_ukr.csv'), nrows=3)
print("=== Перші 3 рядки (preview) ===")
print(df_food_preview)

=== Перші 3 рядки (preview) ===
         date  admin1  admin2            market  market_id  latitude  \
0  2014-03-15     NaN     NaN  National Average       1295       NaN   
1  2014-03-15     NaN     NaN  National Average       1295       NaN   
2  2014-03-15     NaN     NaN  National Average       1295       NaN   

   longitude            category        commodity  commodity_id  unit  \
0        NaN  cereals and tubers      Bread (rye)           374  Loaf   
1        NaN  cereals and tubers  Buckwheat grits           373    KG   
2        NaN  cereals and tubers         Potatoes            83    KG   

  priceflag pricetype currency  price  usdprice  
0    actual    Retail      UAH   4.96      0.52  
1    actual    Retail      UAH   7.44      0.78  
2    actual    Retail      UAH   6.74      0.70  


In [15]:
df_food_preview

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2014-03-15,NaN,NaN,National Average,1295,NaN,NaN,cereals and tubers,Bread (rye),374,Loaf,actual,Retail,UAH,4.96,0.52
1,2014-03-15,NaN,NaN,National Average,1295,NaN,NaN,cereals and tubers,Buckwheat grits,373,KG,actual,Retail,UAH,7.44,0.78
2,2014-03-15,NaN,NaN,National Average,1295,NaN,NaN,cereals and tubers,Potatoes,83,KG,actual,Retail,UAH,6.74,0.70


In [16]:
print(f"\nКолонки: {list(df_food_preview.columns)}")


Колонки: ['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude', 'longitude', 'category', 'commodity', 'commodity_id', 'unit', 'priceflag', 'pricetype', 'currency', 'price', 'usdprice']


In [17]:
df_food_preview.columns

Index(['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude',
       'longitude', 'category', 'commodity', 'commodity_id', 'unit',
       'priceflag', 'pricetype', 'currency', 'price', 'usdprice'],
      dtype='str')

In [18]:
# --- Завантажуємо повний датасет, але тільки потрібні колонки ---
# Нам потрібні: дата, регіон, ринок, категорія товару, товар, ціна в UAH
FOOD_COLS = ['date', 'admin1', 'market', 'category', 'commodity', 'unit', 'currency', 'price']


In [19]:
df_food = pd.read_csv(
    os.path.join(DATA_DIR, 'wfp_food_prices_ukr.csv'),
    usecols=FOOD_COLS,
    parse_dates=['date']  # автоматично розпізнаємо дати
)

In [20]:
print(f"Завантажено: {df_food.shape[0]:,} рядків × {df_food.shape[1]} стовпців")
print(f"\nПеріод даних: {df_food['date'].min().date()} — {df_food['date'].max().date()}")
df_food.head()

Завантажено: 112,259 рядків × 8 стовпців

Період даних: 2014-03-15 — 2026-08-15


,date,admin1,market,category,commodity,unit,currency,price
0,2014-03-15,NaN,National Average,cereals and tubers,Bread (rye),Loaf,UAH,4.96
1,2014-03-15,NaN,National Average,cereals and tubers,Buckwheat grits,KG,UAH,7.44
2,2014-03-15,NaN,National Average,cereals and tubers,Potatoes,KG,UAH,6.74
3,2014-03-15,NaN,National Average,cereals and tubers,Rice,KG,UAH,8.66
4,2014-03-15,NaN,National Average,"meat, fish and eggs",Eggs,10 pcs,UAH,9.49


### 🆕 National Average після 2017 року

У свіжих даних WFP (оновлення 2026) рядки **«National Average»** є лише за **2014–2017**: з 2018 року WFP публікує ціни тільки по регіональних ринках (`admin1` заповнений). Код нижче в ноутбуці бере «ціну по Україні» як `admin1.isna()` — тож без виправлення графіки обірвуться на 2017 році, а «злам 2022» просто не буде видно.

Дораховуємо національне середнє самі — як **просте середнє по всіх ринках** на кожну дату, товар і одиницю — там, де офіційного National Average немає. Це класичне *Split-Apply-Combine* (`groupby` → `mean` → `concat`).

In [21]:
# 🆕 Дорахувати National Average (2018–2026) як середнє по регіональних ринках
def add_national_average(df):
    official_dates = set(df.loc[df["admin1"].isna(), "date"])       # де WFP сам дав National Average
    regional = df[df["admin1"].notna() & ~df["date"].isin(official_dates)]
    computed = (
        regional
        .groupby(["date", "category", "commodity", "unit", "currency"], as_index=False)["price"]
        .mean()
        .assign(market="National Average (розрахунок)")
    )
    return pd.concat([df, computed], ignore_index=True)


In [22]:
df_food = add_national_average(df_food)
national = df_food[df_food["admin1"].isna()]
print("National Average:", national["date"].min().date(), "—", national["date"].max().date(),
      "|", national.groupby("market").size().to_dict())

National Average: 2014-03-15 — 2026-08-15 | {'National Average': 986, 'National Average (розрахунок)': 3787}


In [23]:
# --- Завантажуємо курс гривні ---
df_rates = pd.read_csv(
    os.path.join(DATA_DIR, 'exchange-rates_ukr.csv'),
    usecols=['StartDate', 'Year', 'Months', 'Value'],
    parse_dates=['StartDate']
)
print(df_rates.columns)
# Перейменовуємо колонки для зручності
df_rates.columns = ['date', 'year', 'month', 'uah_per_usd']
print(f"Курс UAH/USD: {df_rates.shape[0]} записів")
df_rates.head()

Index(['StartDate', 'Year', 'Months', 'Value'], dtype='str')
Курс UAH/USD: 431 записів


,date,year,month,uah_per_usd
0,1996-02-01,1996,February,1.881034
1,1996-03-01,1996,March,1.891516
2,1996-04-01,1996,April,1.872667
3,1996-05-01,1996,May,1.841900
4,1996-06-01,1996,June,1.819767


---

## 5. Огляд даних: head, info, describe 🔍 <a id='5-explore'></a>

### Принцип: спочатку зрозумій, потім маніпулюй

Три методи, які треба запускати на **кожному** новому датасеті:

| Метод | Що показує |
|---|---|
| `df.head(n)` | Перші n рядків — візуальне розуміння структури |
| `df.info()` | Технічний зріз: типи, non-null значення, пам'ять |
| `df.describe()` | Описова статистика: mean, std, min, quartiles, max |

Також корисні:
- `df.shape` — (рядки, стовпці)
- `df.dtypes` — типи кожної колонки
- `df.isnull().sum()` — кількість пропущених значень
- `df['col'].value_counts()` — частота кожного значення

In [24]:
# --- Технічний зріз датасету цін на продукти ---
print("=" * 60)
df_food.info()
print("=" * 60)

<class 'pandas.DataFrame'>
RangeIndex: 116046 entries, 0 to 116045
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   date       116046 non-null  datetime64[us]
 1   admin1     111273 non-null  str           
 2   market     116046 non-null  str           
 3   category   116046 non-null  str           
 4   commodity  116046 non-null  str           
 5   unit       116046 non-null  str           
 6   currency   116046 non-null  str           
 7   price      116046 non-null  float64       
dtypes: datetime64[us](1), float64(1), str(6)
memory usage: 13.0 MB


In [25]:
# --- Описова статистика ---
print("=== Описова статистика числових колонок ===")
df_food.describe()

=== Описова статистика числових колонок ===


,date,price
count,116046,116046.000000
mean,2021-03-20 12:35:57.654723,49.079092
min,2014-03-15 00:00:00,1.390000
25%,2019-03-15 00:00:00,15.660000
50%,2021-07-15 00:00:00,28.210000
75%,2023-12-15 00:00:00,59.370000
max,2026-08-15 00:00:00,434.120000
std,NaN,54.416622


In [28]:
# --- Дослідження пропущених значень ---
print("=== Пропущені значення ===")
missing = df_food.isnull().sum()
missing_pct = (missing / len(df_food) * 100).round(1)
print(pd.DataFrame({'Кількість NaN': missing, '% від загального': missing_pct}))
print(f"\n💡 'admin1' (область) відсутня у {missing_pct['admin1']}% рядків — ці записи відносяться до 'National Average'")

=== Пропущені значення ===
           Кількість NaN  % від загального
date                   0               0.0
admin1              4773               4.1
market                 0               0.0
category               0               0.0
commodity              0               0.0
unit                   0               0.0
currency               0               0.0
price                  0               0.0

💡 'admin1' (область) відсутня у 4.1% рядків — ці записи відносяться до 'National Average'


In [30]:
print("=== Унікальні категорії товарів ===")
print(df_food['category'].value_counts())

=== Унікальні категорії товарів ===
category
cereals and tubers       31804
non-food                 26758
meat, fish and eggs      20017
vegetables and fruits    15705
milk and dairy            9614
oil and fats              8715
miscellaneous food        3433
Name: count, dtype: int64


In [32]:
# --- Унікальні значення категоріальних колонок ---
print(f"\n=== Топ-10 товарів за кількістю спостережень ===")
print(df_food['commodity'].value_counts().head(10))


=== Топ-10 товарів за кількістю спостережень ===
commodity
Sugar            3433
Rice             3432
Fuel (diesel)    3423
Fat (salo)       3418
Potatoes         3402
Sour cream       3394
Cabbage          3380
Onions           3378
Beetroots        3366
Meat (pork)      3359
Name: count, dtype: int64


### 🎯 Міні-завдання 1

1. Знайдіть кількість унікальних ринків (`market`) у датасеті
2. Визначте, скільки регіонів (`admin1`) охоплює датасет
3. Яка мінімальна та максимальна ціна в датасеті? Це виглядає реалістично?

> **Очікуваний інсайт:** Значна частина записів — це «National Average» (без прив'язки до конкретної локації). Це важливо враховувати при регіональному аналізі.

In [ ]:
# 🎯 Ваш код тут:


---

## 6. Фільтрація даних: loc та query 🎯 <a id='6-filtering'></a>

### Принцип: відсікай зайве

Думати таблицями — це вміти залишати лише те, що потрібно.

### `.loc[]` — фільтрація через булеві маски

```python
# Синтаксис для Series (1 аргумент — рядки):
s.loc[умова_рядків]

# Синтаксис для DataFrame (2 аргументи — рядки + стовпці):
df.loc[умова_рядків, ['col1', 'col2']]  # вибір рядків І стовпців

# ТІЛЬКИ .loc дозволяє модифікувати дані:
df.loc[df['price'] < 10, 'price_flag'] = 'low'
```

**Як працює під капотом:**
При умові `(df['a'] > 100) & (df['b'] < 700)` Pandas **створює окремий тимчасовий булевий масив** для кожної умови, а потім об'єднує їх через `&`. На 1 млн рядків і 3 умовах — це мільйони тимчасових записів у пам'яті.

### `.query()` — SQL-подібний синтаксис через рушій numexpr

```python
df.query('price > 50 and category == "Vegetables"')
```

**Як працює під капотом:**
Метод `.query()` передає текстовий рядок спеціальній бібліотеці **`numexpr`**, яка компілює та виконує всі логічні операції **«на льоту»** без створення проміжних масивів. Це кардинально зменшує споживання RAM при складних умовах на великих даних.

⚠️ На малих даних (<10k рядків) парсинг рядка в `.query()` дає накладні витрати — тому `.loc` може бути швидшим.

### Архітектурне порівняння: коли що обирати?

| Критерій | `.loc[]` | `.query()` |
|---|---|---|
| **Синтаксис** | Булева маска (повторення імені df) | SQL-рядок (лаконічно) |
| **Пам'ять** | ❌ Тимчасові масиви для кожної умови | ✅ `numexpr` — без зайвих масивів |
| **Швидкість** | Краще на малих даних (<10k рядків) | Краще на великих даних |
| **Вибір стовпців** | ✅ `df.loc[умова, ['a','b']]` | ❌ Завжди всі стовпці |
| **Модифікація даних** | ✅ `df.loc[умова, 'col'] = val` | ❌ Тільки читання |

> 💡 **Правило:** `query()` — для читання великих датасетів зі складними умовами. `loc` — для запису або коли потрібно одночасно відібрати конкретні стовпці.

In [ ]:
# --- Фільтрація: тільки ринки в регіонах (без 'National Average') ---
df_regions = df_food.loc[df_food['admin1'].notna()].copy()

In [34]:
print(f"Записів з регіональною прив'язкою: {len(df_regions):,}")
print(f"Унікальних регіонів: {df_regions['admin1'].nunique()}")
print(df_regions['admin1'].value_counts().head(27))

Записів з регіональною прив'язкою: 111,273
Унікальних регіонів: 26
admin1
Kyiv                         9138
Volynska                     4734
Chernivetska                 4731
Zaporizka                    4692
Cherkaska                    4681
Poltavska                    4676
Mykolaivska                  4662
Khersonska                   4650
Dnipropetrovska              4645
Vinnytska                    4642
Lvivska                      4630
Khmelnytska                  4617
Zakarpatska                  4615
Sumska                       4611
Odeska                       4605
Chernihivska                 4602
Zhytomyrska                  4597
Kirovohradska                4577
Ternopilska                  4574
Rivnenska                    4571
Kharkivska                   4525
Ivano-Frankivska             4506
Donetska                     2513
Luhanska                     2415
Sevastopol                     43
Avtonomna Respublika Krym      21
Name: count, dtype: int64


In [35]:
# --- .loc: вибір рядків І стовпців одночасно ---
# Знайдемо дані по хлібу з 2022 року
mask_bread = df_food['commodity'].str.contains('Bread|Хліб', case=False, na=False)

In [41]:
mask_2022 = df_food['date'].dt.year >= 2022

In [43]:
df_bread_2022 = df_food.loc[mask_bread & mask_2022, ['date', 'admin1', 'commodity', 'price']]
print(f"Записів про хліб з 2022 року: {len(df_bread_2022):,}")
df_bread_2022

Записів про хліб з 2022 року: 7,108


,date,admin1,commodity,price
38231,2020-01-15,Cherkaska,Bread (first grade flour),19.720000
38232,2020-01-15,Cherkaska,Bread (high grade flour),24.740000
38233,2020-01-15,Cherkaska,Bread (rye),20.830000
38234,2020-01-15,Cherkaska,Bread (wheat),12.090000
38272,2020-01-15,Chernihivska,Bread (first grade flour),19.430000
...,...,...,...,...
115974,2026-07-15,NaN,Bread (high grade flour),64.910909
115975,2026-07-15,NaN,Bread (rye),56.489130
116010,2026-08-15,NaN,Bread (first grade flour),55.655909
116011,2026-08-15,NaN,Bread (high grade flour),64.963333


In [40]:
# --- .query() — більш читабельний синтаксис ---
# Знайдемо товари категорії 'cereals and tubers' дорожче 50 UAH
df_expensive_cereals = df_food.query(
    "category == 'cereals and tubers' and price > 50 and currency == 'UAH'"
)
print(f"Дорогі зернові (>50 UAH): {len(df_expensive_cereals):,} записів")
print("\nТоп-5 найдорожчих:")
df_expensive_cereals.nlargest(5, 'price')[['date', 'commodity', 'price', 'admin1']]

Дорогі зернові (>50 UAH): 2,508 записів

Топ-5 найдорожчих:


,date,commodity,price,admin1
69475,2022-08-15,Buckwheat,104.75,Dnipropetrovska
69065,2022-07-15,Buckwheat,104.56,Sumska
69349,2022-08-15,Buckwheat,103.06,Cherkaska
68533,2022-07-15,Buckwheat,101.79,Dnipropetrovska
70010,2022-08-15,Buckwheat,101.49,Sumska


In [44]:
# --- Порівняння продуктивності: .loc vs .query ---
# На великих датасетах query() швидший
import time

start = time.perf_counter()
result_loc = df_food.loc[(df_food['price'] > 30) & (df_food['currency'] == 'UAH')]
time_loc = time.perf_counter() - start

start = time.perf_counter()
result_query = df_food.query("price > 30 and currency == 'UAH'")
time_query = time.perf_counter() - start

print(f".loc   : {time_loc*1000:.2f} ms, результатів: {len(result_loc):,}")
print(f".query : {time_query*1000:.2f} ms, результатів: {len(result_query):,}")
print(f"\n✅ Однаковий результат: {len(result_loc) == len(result_query)}")

.loc   : 17.38 ms, результатів: 54,504
.query : 23.68 ms, результатів: 54,504

✅ Однаковий результат: True


### 🎯 Міні-завдання 2

1. Відфільтруй записи лише для категорії `'vegetables and fruits'` після 2020 року
2. Знайди записи, де ціна перевищує середню ціну всього датасету
3. Використай `.query()` для фільтрації конкретного товару в конкретному регіоні

> **Очікуваний інсайт:** Після 2022 року ціни на овочі та зернові різко зросли — це відображає реальний вплив повномасштабного вторгнення на продовольчу безпеку.

In [ ]:
# 🎯 Ваш код тут:


---

## 7. Групування та агрегації: Split-Apply-Combine 📊 <a id='7-groupby'></a>

### Парадигма: погляд «згори»

Аналітика часто вимагає погляду на агреговані дані. У `pandas` це реалізовано через парадигму **Split → Apply → Combine**:

```
1. SPLIT    → groupby('category')  → розбити на групи
2. APPLY    → .mean()              → порахувати в кожній групі
3. COMBINE  → підсумкова таблиця   → зшити результати назад
```

### Архітектурна деталь: `groupby()` — ліниве виконання 🦥

Виклик `df.groupby('col')` **не виконує жодних обчислень одразу!**

Він лише створює спеціальний **`DataFrameGroupBy`** об'єкт — словник індексів (покажчиків на рядки, які належать до кожної групи). Реальні обчислення стартують тільки коли ти викликаєш агрегаційну функцію (`.mean()`, `.sum()`). Це суттєво економить пам'ять.

### Чотири шляхи на стадії Apply

| Метод | Що робить | Коли використовувати |
|---|---|---|
| `.agg()` | Зменшує групу до одного числа | Статистика: mean, sum, count, max |
| `.transform()` | Зберігає форму — повертає той самий розмір | Нормалізація в межах групи |
| `.filter()` | Відкидає або залишає цілі групи | Видалення рідкісних категорій |
| `.apply()` | Будь-яка довільна функція | Складна логіка, що не вписується в agg |

### Метод `.agg()` — кілька агрегацій одночасно

```python
df.groupby('category').agg(
    середня_ціна=('price', 'mean'),
    макс_ціна=('price', 'max'),
    унікальних_товарів=('commodity', 'nunique')
)
```

In [45]:
# --- Базова агрегація: середня ціна за категорією товару ---
df_uah = df_food.query("currency == 'UAH'")  # тільки гривневі ціни

In [46]:
avg_by_category = (
    df_uah
    .groupby('category')['price']
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
print("=== Середня ціна за категорією (UAH) ===")
print(avg_by_category)

=== Середня ціна за категорією (UAH) ===
category
meat, fish and eggs      107.57
milk and dairy            71.68
oil and fats              70.51
non-food                  43.16
cereals and tubers        24.30
miscellaneous food        21.87
vegetables and fruits     15.02
Name: price, dtype: float64


In [47]:
# --- Множинна агрегація через .agg() ---
stats_by_category = (
    df_uah
    .groupby('category')
    .agg(
        середня_ціна=('price', 'mean'),
        мін_ціна=('price', 'min'),
        макс_ціна=('price', 'max'),
        кількість_спостережень=('price', 'count'),
        унікальних_товарів=('commodity', 'nunique')
    )
    .round(2)
    .sort_values('середня_ціна', ascending=False)
)
print("=== Статистика за категоріями ===")
stats_by_category

=== Статистика за категоріями ===


,середня_ціна,мін_ціна,макс_ціна,кількість_спостережень,унікальних_товарів
category,,,,,
"meat, fish and eggs",107.57,6.35,434.12,20017,7
milk and dairy,71.68,6.81,274.02,9614,3
oil and fats,70.51,12.44,289.97,8715,3
non-food,43.16,2.50,415.06,26758,12
cereals and tubers,24.30,2.09,104.75,31804,14
miscellaneous food,21.87,7.64,64.69,3433,1
vegetables and fruits,15.02,1.39,98.68,15705,5


In [48]:
# --- Аналіз цінових тенденцій по роках ---
# Додаємо колонку 'рік' для групування
df_uah_copy = df_uah.copy()
df_uah_copy['рік'] = df_uah_copy['date'].dt.year

In [49]:
# Середня ціна ХЛІБА по роках
bread_yearly = (
    df_uah_copy
    .query("commodity.str.contains('Bread', case=False)")
    .groupby('рік')['price']
    .agg(
        середня=('mean'),
        медіана=('median')
    )
    .round(2)
)

In [50]:

print("=== Ціна хліба по роках (UAH) ===")
print(bread_yearly)
print(f"\n📈 Зріст ціни хліба з {bread_yearly.index.min()} по {bread_yearly.index.max()}:")
price_start = bread_yearly['середня'].iloc[0]
price_end = bread_yearly['середня'].iloc[-1]
print(f"   {price_start:.2f} → {price_end:.2f} UAH (+{(price_end/price_start - 1)*100:.0f}%)")

=== Ціна хліба по роках (UAH) ===
      середня  медіана
рік                   
2014     5.62     5.46
2015     9.16     9.60
2016    10.14    10.10
2017    11.94    11.88
2018    17.13    17.47
2019    18.54    18.89
2020    19.82    20.46
2021    23.23    23.38
2022    29.61    30.04
2023    33.16    33.44
2024    36.86    36.83
2025    45.27    46.18
2026    56.64    55.24

📈 Зріст ціни хліба з 2014 по 2026:
   5.62 → 56.64 UAH (+908%)


In [53]:
# --- Групування за кількома ключами: рік + категорія ---
# Порівняємо як змінювалися ціни в різних категоріях
df_uah_copy['Рік'] = df_uah_copy['date'].dt.year

In [54]:
yearly_category = (
    df_uah_copy
    .groupby(['Рік', 'category'])['price']
    .mean()
    .round(2)
    .unstack()  # перетворює категорії у стовпці
)
print("=== Середня ціна за категорією по роках ===")
yearly_category.tail(5)  # останні 5 років

=== Середня ціна за категорією по роках ===


category,cereals and tubers,"meat, fish and eggs",milk and dairy,miscellaneous food,non-food,oil and fats,vegetables and fruits
Рік,,,,,,,
2022,30.11,123.67,86.88,31.65,45.45,86.04,20.82
2023,30.05,146.02,100.22,35.08,50.81,99.46,24.23
2024,30.29,166.01,115.55,32.04,56.42,101.85,20.66
2025,35.00,185.10,136.68,31.83,67.26,123.67,29.31
2026,43.32,204.78,144.55,29.42,70.07,152.09,23.20


In [55]:
# --- Демонстрація .transform(): нормалізація всередині групи ---
# transform() повертає Series того ж розміру, що й вхідний DataFrame
# Корисно для: нормалізація, заповнення NaN груповим середнім, Z-score

df_uah_copy2 = df_uah.copy()

# Розраховуємо середню ціну в кожній групі (категорії) та додаємо як нову колонку
df_uah_copy2['середня_по_категорії'] = (
    df_uah_copy2.groupby('category')['price'].transform('mean').round(2)
)

In [56]:
# Відхилення від середнього в категорії
df_uah_copy2['відхилення_%'] = (
    (df_uah_copy2['price'] / df_uah_copy2['середня_по_категорії'] - 1) * 100
).round(1)

print("=== .transform() — ціна відносно середньої по категорії ===")
print("Перші 8 рядків з новими колонками:")
print(df_uah_copy2[['category', 'commodity', 'price', 'середня_по_категорії', 'відхилення_%']].head(8))
print("\n💡 transform() зберігає розмір DataFrame — кожен рядок отримує агрегат своєї групи")

=== .transform() — ціна відносно середньої по категорії ===
Перші 8 рядків з новими колонками:
              category              commodity  price  середня_по_категорії  \
0   cereals and tubers            Bread (rye)   4.96                 24.30   
1   cereals and tubers        Buckwheat grits   7.44                 24.30   
2   cereals and tubers               Potatoes   6.74                 24.30   
3   cereals and tubers                   Rice   8.66                 24.30   
4  meat, fish and eggs                   Eggs   9.49                107.57   
5  meat, fish and eggs  Meat (chicken, whole)  21.03                107.57   
6  meat, fish and eggs  Meat (mixed, sausage)  33.04                107.57   
7  meat, fish and eggs            Meat (pork)  44.78                107.57   

   відхилення_%  
0         -79.6  
1         -69.4  
2         -72.3  
3         -64.4  
4         -91.2  
5         -80.4  
6         -69.3  
7         -58.4  

💡 transform() зберігає розмір DataFrame

In [57]:
# --- Демонстрація .filter(): відкидаємо цілі групи ---
# filter() відкидає або залишає цілі групи на основі логічного критерію

# Залишимо лише ті категорії, де кількість спостережень > 1000
df_uah_filtered = (
    df_uah
    .groupby('category')
    .filter(lambda group: len(group) > 1000)
)

print("=== .filter() — лише категорії з >1000 спостережень ===")
print("До filter:")
print(df_uah['category'].value_counts())
print("\nПісля filter (рідкісні категорії видалено):")
print(df_uah_filtered['category'].value_counts())
print(f"\nВидалено рядків: {len(df_uah) - len(df_uah_filtered):,}")

=== .filter() — лише категорії з >1000 спостережень ===
До filter:
category
cereals and tubers       31804
non-food                 26758
meat, fish and eggs      20017
vegetables and fruits    15705
milk and dairy            9614
oil and fats              8715
miscellaneous food        3433
Name: count, dtype: int64

Після filter (рідкісні категорії видалено):
category
cereals and tubers       31804
non-food                 26758
meat, fish and eggs      20017
vegetables and fruits    15705
milk and dairy            9614
oil and fats              8715
miscellaneous food        3433
Name: count, dtype: int64

Видалено рядків: 0


### 🎯 Міні-завдання 3

1. Знайдіть топ-5 найдорожчих товарів (за середньою ціною) в 2023 році
2. Порівняйте середню ціну `Potatoes` до і після 2022 року
3. Скільки унікальних товарів є в кожній категорії?

> **Очікуваний інсайт:** Ціни на товари після 2022 року зросли нерівномірно — деякі категорії (олія, м'ясо) показали більш різкий стрибок, ніж інші.

In [ ]:
# 🎯 Ваш код тут:


---

## 8. Робота з датами (datetime) 📅 <a id='8-datetime'></a>

### Часові ряди — специфічний тип табличних даних

Звичайний текст не дозволяє аналізувати час. Після конвертації у `datetime64` з'являється потужний аксесор **`.dt`**, який дозволяє витягувати будь-яку складову дати:

| Аксесор | Повертає |
|---|---|
| `.dt.year` | Рік |
| `.dt.month` | Місяць (1–12) |
| `.dt.month_name()` | Назва місяця |
| `.dt.dayofweek` | День тижня (0=Пн, 6=Нд) |
| `.dt.quarter` | Квартал (1–4) |
| `.dt.to_period('M')` | Перетворення в Period (рік-місяць) |

### Ресемплінг (resample)

Метод `.resample()` — це `groupby` для часових рядів. Дозволяє агрегувати дані по часових інтервалах (тижень, місяць, рік).

In [58]:
# --- Перевірка типу дати ---
print(f"Тип колонки 'date': {df_food['date'].dtype}")
print(f"\nПриклад значення: {df_food['date'].iloc[0]}")
print(f"Тип одного значення: {type(df_food['date'].iloc[0])}")

Тип колонки 'date': datetime64[us]

Приклад значення: 2014-03-15 00:00:00
Тип одного значення: <class 'pandas.Timestamp'>


In [59]:
# --- Аксесор .dt: розбиваємо дату на складові ---
df_time = df_uah.copy()
df_time['рік'] = df_time['date'].dt.year
df_time['місяць'] = df_time['date'].dt.month
df_time['назва_місяця'] = df_time['date'].dt.month_name()
df_time['квартал'] = df_time['date'].dt.quarter

print("=== Результат розбивки дати ===")
df_time[['date', 'рік', 'місяць', 'назва_місяця', 'квартал']].drop_duplicates().head(8)

=== Результат розбивки дати ===


,date,рік,місяць,назва_місяця,квартал
0,2014-03-15,2014,3,March,1
591,2014-04-15,2014,4,April,2
1164,2014-05-15,2014,5,May,2
1719,2014-06-15,2014,6,June,2
2273,2014-07-15,2014,7,July,3
2829,2014-08-15,2014,8,August,3
3347,2014-09-15,2014,9,September,3
3862,2014-10-15,2014,10,October,4


In [60]:
# --- Аналіз курсу гривні в часі ---
print("=== Курс UAH/USD по роках ===")
df_rates['рік'] = df_rates['date'].dt.year

yearly_rate = (
    df_rates
    .groupby('рік')['uah_per_usd']
    .mean()
    .round(2)
)
print(yearly_rate.tail(15))

=== Курс UAH/USD по роках ===
рік
2012     7.99
2013     7.99
2014    11.89
2015    21.84
2016    25.55
2017    26.60
2018    27.20
2019    25.85
2020    26.96
2021    27.29
2022    32.34
2023    36.57
2024    40.15
2025    41.69
2026    42.69
Name: uah_per_usd, dtype: float64


In [61]:
# --- Ресемплінг: середня ціна хліба по місяцях ---
df_bread = (
    df_uah
    .query("commodity.str.contains('Bread', case=False) and admin1.isna()")
    [['date', 'commodity', 'price']]
    .set_index('date')
    .sort_index()
)

# Ресемплінг по кварталах
bread_quarterly = (
    df_bread
    .resample('QE')  # QE = кінець кварталу
    ['price']
    .mean()
    .round(2)
    .reset_index()
)
bread_quarterly.columns = ['квартал', 'середня_ціна_хліба']
print("=== Середня ціна хліба по кварталах ===")
print(bread_quarterly.tail(12))

=== Середня ціна хліба по кварталах ===
      квартал  середня_ціна_хліба
39 2023-12-31               34.00
40 2024-03-31               34.63
41 2024-06-30               35.49
42 2024-09-30               37.88
43 2024-12-31               39.84
44 2025-03-31               41.50
45 2025-06-30               43.17
46 2025-09-30               44.41
47 2025-12-31               51.97
48 2026-03-31               53.86
49 2026-06-30               57.71
50 2026-09-30               59.18


In [62]:
# --- Пошук аномалій у часовому ряді ---
# Знайдемо квартали з найбільшим зростанням ціни
bread_quarterly['зміна_%'] = (
    bread_quarterly['середня_ціна_хліба'].pct_change() * 100
).round(1)

print("=== Квартали з найбільшим зростанням цін на хліб ===")
print(bread_quarterly.nlargest(5, 'зміна_%')[['квартал', 'середня_ціна_хліба', 'зміна_%']])
print("\n💡 Знайдіть квартали, що збігаються з ключовими подіями в Україні")

=== Квартали з найбільшим зростанням цін на хліб ===
      квартал  середня_ціна_хліба  зміна_%
5  2015-06-30                9.59     23.9
4  2015-03-31                7.74     23.4
47 2025-12-31               51.97     17.0
3  2014-12-31                6.27     12.8
32 2022-03-31               27.53     10.8

💡 Знайдіть квартали, що збігаються з ключовими подіями в Україні


### 🎯 Міні-завдання 4

1. Визначте, в якому місяці (1–12) ціни на овочі найвищі впродовж року
2. Порівняйте середній курс UAH/USD до і після 2022 року
3. Знайдіть момент (місяць/рік), коли відбулося найбільше стрибкоподібне зростання курсу

> **Очікуваний інсайт:** Курс гривні корелює з цінами на імпортовані товари — девальвація безпосередньо відображається у споживчих цінах.

In [ ]:
# 🎯 Ваш код тут:


---

## 9. Об'єднання датасетів: merge та concat 🔗 <a id='9-merge'></a>

### Це найважливіша навичка! 🔥

У реальному світі дані розкидані по різних таблицях. Вміння їх зшивати — ключова аналітична навичка.

### Горизонтальне об'єднання: `pd.merge()` → аналог SQL JOIN

| `how=` | Аналог SQL | Поведінка |
|---|---|---|
| `'inner'` | INNER JOIN | Тільки рядки, що є в обох таблицях |
| `'left'` | LEFT JOIN | Всі рядки з лівої + збіги з правої |
| `'right'` | RIGHT JOIN | Всі рядки з правої + збіги з лівої |
| `'outer'` | FULL OUTER JOIN | Всі рядки з обох, NaN де немає пари |

### Вертикальне об'єднання: `pd.concat()` → «поставити таблиці одна під одною»

```python
pd.concat([df_2022, df_2023], ignore_index=True)  # склеїти по рядках
```

In [63]:
# --- Підготовка: середня ціна хліба по місяцях (National Average) ---
bread_monthly = (
    df_food
    .query(
        "commodity.str.contains('Bread', case=False) "
        "and admin1.isna() "
        "and currency == 'UAH'"
    )
    .assign(
        рік=lambda x: x['date'].dt.year,
        місяць=lambda x: x['date'].dt.month
    )
    .groupby(['рік', 'місяць'])['price']
    .mean()
    .round(2)
    .reset_index()
)
bread_monthly.columns = ['рік', 'місяць', 'ціна_хліба_uah']
print(f"Місячних записів про ціну хліба: {len(bread_monthly)}")
bread_monthly.tail()

Місячних записів про ціну хліба: 138


,рік,місяць,ціна_хліба_uah
133,2026,4,56.67
134,2026,5,57.94
135,2026,6,58.50
136,2026,7,58.84
137,2026,8,59.52


In [64]:
# --- Підготовка: курс гривні по місяцях ---
rates_monthly = df_rates[['date', 'year', 'uah_per_usd']].copy()
rates_monthly.columns = ['date', 'рік', 'курс_uah_usd']
rates_monthly['місяць'] = rates_monthly['date'].dt.month
rates_monthly = rates_monthly[['рік', 'місяць', 'курс_uah_usd']]
print(f"Місячних записів курсу: {len(rates_monthly)}")
rates_monthly.tail()

Місячних записів курсу: 431


,рік,місяць,курс_uah_usd
426,2021,1,27.286189
427,2022,1,32.342302
428,2023,1,36.573807
429,2024,1,40.152084
430,2025,1,41.689072


In [65]:
# --- MERGE: об'єднуємо ціни хліба та курс UAH/USD ---
# Ключ злиття: рік + місяць
df_bread_rates = pd.merge(
    bread_monthly,        # ліва таблиця
    rates_monthly,        # права таблиця
    on=['рік', 'місяць'], # спільні ключі
    how='inner'           # тільки записи, що є в обох
)

print(f"=== Результат злиття ===")
print(f"Рядків: {len(df_bread_rates)} (перетин дат)")
df_bread_rates.tail(8)

=== Результат злиття ===
Рядків: 151 (перетин дат)


,рік,місяць,ціна_хліба_uah,курс_uah_usd
143,2025,6,43.78,41.579300
144,2025,7,44.08,41.794100
145,2025,8,44.32,41.441900
146,2025,9,44.83,41.333333
147,2025,10,51.55,41.627100
148,2025,11,51.99,42.104000
149,2025,12,52.38,42.199900
150,2026,1,53.23,42.686333


In [66]:
# --- Аналіз: ціна хліба в доларах через злиття ---
df_bread_rates['ціна_хліба_usd'] = (
    df_bread_rates['ціна_хліба_uah'] / df_bread_rates['курс_uah_usd']
).round(3)

print("=== Ціна хліба в UAH та USD по роках ===")
yearly_comparison = (
    df_bread_rates
    .groupby('рік')[['ціна_хліба_uah', 'курс_uah_usd', 'ціна_хліба_usd']]
    .mean()
    .round(3)
)
print(yearly_comparison)
print("""
💡 Інсайт: навіть коли ціна в гривні зростає, ціна в доларах може
   залишатися стабільною або навіть падати — через девальвацію.
   Це показує реальну купівельну спроможність населення.""")

=== Ціна хліба в UAH та USD по роках ===
      ціна_хліба_uah  курс_uah_usd  ціна_хліба_usd
рік                                               
2014           5.625        12.599           0.449
2015           8.816        21.845           0.405
2016          10.106        25.551           0.396
2017          11.765        26.671           0.441
2018          17.140        27.789           0.617
2019          18.401        25.846           0.714
2020          19.781        26.958           0.735
2021          22.984        27.286           0.843
2022          29.279        32.342           0.912
2023          32.942        36.574           0.901
2024          36.599        40.152           0.911
2025          44.659        41.690           1.071
2026          53.230        42.686           1.247

💡 Інсайт: навіть коли ціна в гривні зростає, ціна в доларах може
   залишатися стабільною або навіть падати — через девальвацію.
   Це показує реальну купівельну спроможність населення.


In [67]:
# --- CONCAT: вертикальне об'єднання двох датасетів ---
# Розіб'ємо дані на «до 2022» та «після 2022» і знову склеїмо
df_before_2022 = df_bread_rates.query("рік < 2022").copy()
df_after_2022 = df_bread_rates.query("рік >= 2022").copy()

df_before_2022['епоха'] = 'до 2022'
df_after_2022['епоха'] = 'після 2022'

df_combined = pd.concat([df_before_2022, df_after_2022], ignore_index=True)

print(f"до 2022: {len(df_before_2022)} рядків")
print(f"після 2022: {len(df_after_2022)} рядків")
print(f"після concat: {len(df_combined)} рядків")

# Порівняння середніх по епохах
print("\n=== Порівняння між епохами ===")
df_combined.groupby('епоха')[['ціна_хліба_uah', 'курс_uah_usd', 'ціна_хліба_usd']].mean().round(3)

до 2022: 94 рядків
після 2022: 57 рядків
після concat: 151 рядків

=== Порівняння між епохами ===


,ціна_хліба_uah,курс_uah_usd,ціна_хліба_usd
епоха,,,
до 2022,14.336,24.312,0.576
після 2022,36.174,37.777,0.954


In [73]:
# --- Демонстрація різних типів JOIN ---
# Підготуємо маленькі датасети для наочності
df_prices_small = pd.DataFrame({
    'рік': [2019, 2020, 2021, 2022],
    'ціна_хліба': [12.5, 15.0, 18.3, 26.7]
})
df_rates_small = pd.DataFrame({
    'рік': [2020, 2021, 2022, 2023],  # 2019 відсутній, 2023 — зайвий
    'курс': [26.9, 27.3, 32.3, 36.8]
})

print("INNER JOIN (перетин — тільки 2020-2022):")
print(pd.merge(df_prices_small, df_rates_small, on='рік', how='inner'))

print("\nLEFT JOIN (всі роки лівої таблиці, 2019 без курсу):")
print(pd.merge(df_prices_small, df_rates_small, on='рік', how='left'))

print("\nOUTER JOIN (всі роки, NaN де немає пари):")
print(pd.merge(df_prices_small, df_rates_small, on='рік', how='outer'))

INNER JOIN (перетин — тільки 2020-2022):
    рік  ціна_хліба  курс
0  2020        15.0  26.9
1  2021        18.3  27.3
2  2022        26.7  32.3

LEFT JOIN (всі роки лівої таблиці, 2019 без курсу):
    рік  ціна_хліба  курс
0  2019        12.5   NaN
1  2020        15.0  26.9
2  2021        18.3  27.3
3  2022        26.7  32.3

OUTER JOIN (всі роки, NaN де немає пари):
    рік  ціна_хліба  курс
0  2019        12.5   NaN
1  2020        15.0  26.9
2  2021        18.3  27.3
3  2022        26.7  32.3
4  2023         NaN  36.8


### 🎯 Фінальне завдання

Проведіть комплексний аналіз:

1. Завантажте `global-market-monitor.csv` та відфільтруйте дані тільки по Україні
2. Знайдіть, які продукти показали найбільший річний приріст ціни (`YoYChangeMonth`)
3. Об'єднайте ці дані з курсом UAH/USD через `pd.merge()` по даті/року
4. Дайте відповідь: **чи корелює девальвація гривні зі зростанням цін на продукти?**

> **Очікуваний інсайт:** Аналіз злитих датасетів дозволяє побачити макроекономічні залежності, які не видно з жодної таблиці окремо — це і є справжня сила pandas як аналітичного інструменту.

In [68]:
# --- Крок 1: Завантажуємо глобальний моніторинг ринку ---
df_global = pd.read_csv(
    os.path.join(DATA_DIR, 'global-market-monitor.csv'),
    usecols=['Date', 'CountryName', 'MainStapleFood', 'YoYChangeMonth', 'PriceTrendMonth'],
    parse_dates=['Date']
)

# Відфільтровуємо Україну
df_ukraine_global = (
    df_global
    .query("CountryName.str.strip() == 'Ukraine'")
    .copy()
)
df_ukraine_global.columns = ['date', 'країна', 'продукт', 'зміна_рік_%', 'тренд']
df_ukraine_global['продукт'] = df_ukraine_global['продукт'].str.strip()

print(f"Записів по Україні: {len(df_ukraine_global)}")
print(f"Продукти: {df_ukraine_global['продукт'].unique()}")
df_ukraine_global.head()

Записів по Україні: 138
Продукти: <ArrowStringArray>
['Pasta', 'Oil (sunflower)', 'Potatoes']
Length: 3, dtype: str


,date,країна,продукт,зміна_рік_%,тренд
3796,2022-02-01,Ukraine,Pasta,17.0,Normal
3797,2022-02-01,Ukraine,Oil (sunflower),29.0,Negative
3798,2022-02-01,Ukraine,Potatoes,-21.0,Negative
4028,2022-03-01,Ukraine,Potatoes,-6.0,High
4029,2022-03-01,Ukraine,Pasta,14.0,Moderate


In [69]:
# --- Крок 2: Найбільший річний приріст цін ---
worst_increases = (
    df_ukraine_global
    .dropna(subset=['зміна_рік_%'])
    .nlargest(10, 'зміна_рік_%')
    [['date', 'продукт', 'зміна_рік_%', 'тренд']]
)
print("=== Топ-10 найбільших річних стрибків цін ===")
worst_increases

=== Топ-10 найбільших річних стрибків цін ===


,date,продукт,зміна_рік_%,тренд
9775,2024-11-01,Potatoes,152.0,High
9928,2024-12-01,Potatoes,152.0,N/A
10082,2025-01-01,Potatoes,152.0,N/A
8737,2024-04-01,Potatoes,140.0,N/A
8901,2024-05-01,Potatoes,140.0,N/A
5382,2022-09-01,Pasta,43.0,High
5595,2022-10-01,Pasta,43.0,N/A
5816,2022-11-01,Pasta,43.0,N/A
10407,2025-03-01,Oil (sunflower),39.0,Negative
10578,2025-04-01,Oil (sunflower),39.0,N/A


In [70]:
# --- Крок 3: Об'єднання з курсом валют ---
df_ukraine_global['рік'] = df_ukraine_global['date'].dt.year
df_ukraine_global['місяць'] = df_ukraine_global['date'].dt.month

df_merged_full = pd.merge(
    df_ukraine_global,
    rates_monthly,
    on=['рік', 'місяць'],
    how='left'
)

print(f"Після злиття: {len(df_merged_full)} рядків")
print(f"Пропущених курсів: {df_merged_full['курс_uah_usd'].isna().sum()}")
df_merged_full.dropna(subset=['зміна_рік_%', 'курс_uah_usd'], inplace=True)
df_merged_full.tail(6)

Після злиття: 156 рядків
Пропущених курсів: 0


,date,країна,продукт,зміна_рік_%,тренд,рік,місяць,курс_uah_usd
140,2025-08-01,Ukraine,Pasta,19.0,N/A,2025,8,41.441900
147,2025-11-01,Ukraine,Potatoes,-13.0,N/A,2025,11,42.104000
148,2025-11-01,Ukraine,Pasta,9.0,N/A,2025,11,42.104000
149,2025-11-01,Ukraine,Oil (sunflower),33.0,N/A,2025,11,42.104000
153,2026-01-01,Ukraine,Oil (sunflower),20.0,N/A,2026,1,42.686333
154,2026-01-01,Ukraine,Potatoes,-51.0,N/A,2026,1,42.686333


In [72]:
# --- Крок 4: Кореляція курсу та цін ---
correlation = df_merged_full[['зміна_рік_%', 'курс_uah_usd']].corr()
print("=== Кореляція: зміна ціни (%) ~ курс UAH/USD ===")
print(correlation)

corr_value = correlation.loc['зміна_рік_%', 'курс_uah_usd']

print(f"""
📊 Кореляція = {corr_value:.3f}

Інтерпретація:
  • > 0.7 → сильна пряма кореляція (девальвація → дорожче)
  • 0.3–0.7 → помірна кореляція
  • < 0.3 → слабка кореляція

💡 Середня ціна за курсовими діапазонами:
""")

# Групуємо по курсовим діапазонам
bins = [0, 20, 27, 32, 40, 100]
labels = ['<20', '20-27', '27-32', '32-40', '>40']
df_merged_full['курс_діапазон'] = pd.cut(df_merged_full['курс_uah_usd'], bins=bins, labels=labels)
print(df_merged_full.groupby('курс_діапазон')['зміна_рік_%'].mean().round(1))

=== Кореляція: зміна ціни (%) ~ курс UAH/USD ===
              зміна_рік_%  курс_uah_usd
зміна_рік_%      1.000000      0.238443
курс_uah_usd     0.238443      1.000000

📊 Кореляція = 0.238

Інтерпретація:
  • > 0.7 → сильна пряма кореляція (девальвація → дорожче)
  • 0.3–0.7 → помірна кореляція
  • < 0.3 → слабка кореляція

💡 Середня ціна за курсовими діапазонами:

курс_діапазон
27-32    10.2
32-40    13.5
>40      32.8
Name: зміна_рік_%, dtype: float64


---

## 🏁 Підсумок уроку

### Що ми навчились думати:

```
ДАНІ  →  РОЗУМІННЯ  →  ТРАНСФОРМАЦІЯ  →  ІНСАЙТ
 CSV      .info()        groupby()      кореляція
         describe()       merge()       тренди
```

### Ключові концепції:

| Концепція | Суть |
|---|---|
| **DataFrame** | Модель реального світу: рядки = спостереження, стовпці = атрибути |
| **Series** | 1D масив з індексом — базовий блок DataFrame |
| **Data Alignment** | Операції зіставляють дані за мітками, а не позицією |
| **Category dtype** | Економія до 90% пам'яті для повторюваних текстів |
| **loc vs query** | `loc` — булева маска; `query` — SQL-подібний рядок |
| **groupby** | Split → Apply → Combine |
| **merge vs concat** | Горизонтальне (JOIN) vs вертикальне (склеювання) |
| **.dt accessor** | Витягування складових дати з datetime64 |

### Наступний крок

У **Уроці 52 (частина 2)** ми візуалізуємо ці дані через **Matplotlib та Plotly**

---
*📚 Урок 52 — Модуль 3: Data Analysis & Visualization*